# 11 · A code becomes a projective plane

**Can one seven-bit pattern explain a code, its dual, and the motion of a finite plane?**

Begin with the polynomial $g(x)=1+x+x^3$. Its shifted coefficients generate a
binary Hamming code. Its constraints generate another code. Then seven triples
become the lines of the Fano plane—and the same cubic constructs an eight-element
field whose multiplication moves those lines.

Follow the pictures first, then inspect the definitions beside them. Pause before
each explanation and predict what survives a shift. All constructions are finite
and editable. Smooth paths connect captured exact states; intermediate positions
do not define new binary coefficients or new projective incidences.

**Run:** use the `python3`/venv instructions in [README.md](README.md), install
`python3 -m pip install -e '.[notebooks]'`, select that environment's kernel, then
**Restart Kernel and Run All Cells**. Nothing is downloaded by this notebook.
Exports go to `build/notebooks/cyclic-code/`; the final cells embed two actual MP4s.

In [ ]:
from math import pi, sqrt
from kaleion import Arrangement, Collection, F, Motion, Workspace, cos, param, sin

In [ ]:
from collections import Counter
from itertools import combinations
from pathlib import Path
import json
import sys
import numpy as np
import plotly.io as pio
from IPython.display import Markdown, Video, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").exists())
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))
from kaleion.viewers.plotly import animation_figure
from kaleion.viewers.video import write_mp4
from lesson_views import COLORS, replay, save_figures
from code_views import binary_panels, fano_gallery, linked_field_motion

pio.renderers.default = "plotly_mimetype+notebook"
OUTPUT = ROOT / "build/notebooks/cyclic-code"
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURES = {}

def sampled(transition, name, label, steps=25):
    times = np.linspace(0, 1, steps)
    return ([transition.frame(name, float(t)) for t in times],
            [f"{label} · {t:.0%}" for t in times])

def compact_workspace(workspace, filename):
    payload = json.dumps(json.loads(workspace.to_json()), separators=(",", ":"), allow_nan=False)
    (OUTPUT / filename).write_text(payload)
    return Workspace.from_json(payload)

In [ ]:
# Both degree-three irreducible cubics over F_2 work. Restart and rerun after edits.
PARAMETERS = {"generator": 11, "message": 5, "error_mask": 4}
# generator = 11: 1+x+x^3; generator = 13: 1+x^2+x^3.
# error_mask = 0 means no error; 1,2,4,...,64 are single errors; try 3 for two.
assert PARAMETERS["generator"] in (11, 13), "Use 11 or 13 here; broken cubics have their own challenge below"
assert 0 <= PARAMETERS["message"] < 16
assert 0 <= PARAMETERS["error_mask"] < 128

## A polynomial becomes a matrix

All binary vectors use **low degree first**: $(c_0,\ldots,c_6)$ means
$c_0+c_1x+\cdots+c_6x^6$. For compact labels we pack this as
$\sum c_j2^j$; the integer label 11 therefore means $(1,1,0,1,0,0,0)$.
Packed labels do **not** turn polynomial addition into ordinary integer addition.
Our recipes explicitly add coefficients modulo two.

In $\mathbf F_2[x]/(x^7-1)$, multiplication by $x$ shifts the seven slots cyclically.
Make four copies of $g$, shift copy $r$ by $r$, and flatten the rings into rows.
Notice that the zero coefficients move too. A copy has its own occurrence identity;
shifting or changing its placement preserves that identity.

In [ ]:
def bit(word, degree):
    """Low-degree-first binary coefficients, using exact integer arithmetic."""
    return (word // (2 ** degree)) % 2


def polynomial_product(left, right, left_width, right_width):
    """Bounded binary convolution; works on integers or symbolic expressions."""
    return sum(
        (sum(bit(left, i) * bit(right, degree-i)
             for i in range(left_width) if 0 <= degree-i < right_width) % 2)
        * 2 ** degree
        for degree in range(left_width + right_width - 1)
    )


def coefficients(packed):
    return Collection.sequence(7, start=0, name="Seven coefficient slots").with_values(
        bit(packed, F.key))


def generator_rows(polynomial, dimension):
    """Copies and cyclic shifts retain every coefficient occurrence, including 0."""
    copies = Collection.grid(dimension, 7, axes=("row", "seed"),
                             values=polynomial.bind(on=F.seed))
    shifted = copies.annotate(degree=(F.seed + F.row) % 7)
    return copies, shifted

In [ ]:
def generator_placements(copies, matrix):
    def rings(items, degree):
        return items.arrange(x=(2+F.row)*cos(2*pi*degree/7),
                             y=(2+F.row)*sin(2*pi*degree/7), z=F.row)
    return (rings(copies, F.seed), rings(matrix, F.degree),
            matrix.arrange(x=F.degree, y=-F.row, z=0))


def generator_turn():
    angle = 2*pi*F.sz*param("time")/7
    return Motion.custom(F.sx*cos(angle)-F.sy*sin(angle),
                         F.sx*sin(angle)+F.sy*cos(angle), F.sz)

In [ ]:
g = coefficients(param("generator"))
copies, G = generator_rows(g, 4)
unshifted_rings, shifted_rings, matrix_G = generator_placements(copies, G)

generator_workspace = Workspace({"coefficients": unshifted_rings.where(F.value == 1)}, PARAMETERS)
turn = generator_workspace.set("coefficients", shifted_rings.where(F.value == 1), motion=generator_turn())
unfold = generator_workspace.set("coefficients", matrix_G.where(F.value == 1),
                                 motion=Motion.arc(height=1.3, axis=2, dimension=3))
generator_workspace.capture("The four cyclic shifts form G; every zero slot remains present.")
fold_back = generator_workspace.undo()
assert generator_workspace.can_redo

generator_frames, generator_labels = [], []
for transition, label in ((turn, "Shift each copy"), (unfold, "Read the rows of G"),
                           (fold_back, "Undo the unfolding")):
    frames, labels = sampled(transition, "coefficients", label)
    generator_frames.extend(frames)
    generator_labels.extend(labels)
assert unfold.start.results["coefficients"].source.ids == unfold.end.results["coefficients"].source.ids
generator_plot = animation_figure(generator_frames, labels=generator_labels,
                                  title="A polynomial becomes a matrix", duration=70)
generator_plot.update_layout(scene_camera_eye=dict(x=1.3, y=1.7, z=1.8))
FIGURES["generator-motion"] = generator_plot
generator_plot.show()

Now choose four message bits $m_0,\ldots,m_3$. At output degree $j$, count the
selected rows contributing a 1, then take the parity:

$$n_{m,j}=\#\{r:m_r=1\;\land\;G_{rj}=1\},\qquad c_{m,j}=n_{m,j}\bmod2.$$

This is both $c=mG$ and the coefficient convolution for $c(x)=m(x)g(x)$.
With $\deg m<4$ and $\deg g=3$, the product already has degree below seven.
The integer count and its parity are different derived objects. A zero parity
can hide two contributors; a zero count has none.

In [ ]:
def span(matrix, dimension):
    terms = Collection.grid(2 ** dimension, dimension, 7,
                            axes=("message", "row", "degree"),
                            values=bit(F.message, F.row) * matrix.bind(
                                on=(F.row, F.degree), key=(F.row, F.degree)))
    overlaps = terms.where(F.value == 1).group_by(F.message, F.degree).count()
    bits = overlaps.with_values(F.value % 2)
    words = bits.group_by(F.message).sum(value=F.value * 2 ** F.degree)
    weights = bits.where(F.value == 1).group_by(F.message).count()
    return dict(terms=terms, overlaps=overlaps, bits=bits, words=words, weights=weights)

In [ ]:
code = span(G, 4)
code_workspace = Workspace({**code, "G": matrix_G}, PARAMETERS)
assert not code_workspace.state.errors
c = code_workspace.state.results
print("Codewords by message:", c["words"].values.tolist())
print("Weight distribution:", dict(sorted(Counter(c["weights"].values).items())))
print("Selected message:", PARAMETERS["message"])
chosen = c["overlaps"].fields["message"] == PARAMETERS["message"]
print("Integer overlaps:", c["overlaps"].values[chosen].tolist())
print("Output parities: ", c["bits"].values[chosen].tolist())
assert len(set(c["words"].values)) == 16
assert Counter(c["weights"].values) == {0: 1, 3: 7, 4: 7, 7: 1}

## The constraints become generators

Find the unique polynomial $h$ of degree at most four with $g h=x^7+1$.
We inspect all 32 candidates and retain the successful one, with its contributor
evidence. This is a bounded factor search, not a general polynomial solver.

Reverse its five coefficients to form $g^\perp(x)=x^4h(x^{-1})$. For the default:

$$h=1+x+x^2+x^4,\qquad g^\perp=1+x^2+x^3+x^4.$$

Three shifted copies of $g^\perp$ form a matrix $H$. Its rows generate the dual
code $C^\perp$, and check the original code. The overlaps in $G H^T$ are **even**;
they need not be zero as ordinary integer counts.

In [ ]:
def dual_polynomial(polynomial):
    packed = polynomial.sum(value=F.value * 2 ** F.key).scalar()
    candidates = Collection.sequence(32, start=0, name="Degree-at-most-four quotients")
    candidates = candidates.annotate(product=polynomial_product(packed, F.value, 4, 5))
    factors = candidates.where(F.product == 129)  # x^7 + 1
    quotient = factors.group_by().coverage().unique()
    # Reverse degrees 0..4 in seven padded slots. Degrees 5 and 6 read zeros.
    reciprocal = coefficients(quotient.scalar()).with_values(
        bit(quotient.scalar(), (4 - F.key) % 7))
    return factors, quotient, reciprocal


def cross_parities(left, right, left_dimension, right_dimension):
    pairs = Collection.grid(left_dimension, right_dimension, 7,
                            axes=("left", "right", "degree"),
                            values=left.bind(on=(F.left, F.degree), key=(F.row, F.degree))
                            * right.bind(on=(F.right, F.degree), key=(F.row, F.degree)))
    overlaps = pairs.where(F.value == 1).group_by(F.left, F.right).count()
    return overlaps, overlaps.with_values(F.value % 2)


def parity_checks(matrix, dimension):
    trials = Collection.grid(128, dimension, 7, axes=("word", "row", "degree"),
                             values=bit(F.word, F.degree) * matrix.bind(
                                 on=(F.row, F.degree), key=(F.row, F.degree)))
    overlaps = trials.where(F.value == 1).group_by(F.word, F.row).count()
    parities = overlaps.with_values(F.value % 2)
    syndromes = parities.group_by(F.word).sum(value=F.value * 2 ** F.row)
    kernel = syndromes.where(F.value == 0).select().with_values(F.word)
    return dict(overlaps=overlaps, parities=parities, syndromes=syndromes, kernel=kernel)

In [ ]:
factors, quotient, dual_g = dual_polynomial(g)
_, H = generator_rows(dual_g, 3)
dual = span(H, 3)
overlaps, cross = cross_parities(G, H, 4, 3)
checks_H = parity_checks(H, 3)
checks_G = parity_checks(G, 4)

roots = {"quotient": quotient, "g_perp": dual_g, "G": matrix_G,
         "H": H.arrange(x=F.degree, y=-F.row),
         "overlaps": overlaps, "cross": cross,
         "code": code["words"], "dual": dual["words"],
         "kernel_H": checks_H["kernel"], "kernel_G": checks_G["kernel"]}
algebra_workspace = Workspace(roots, PARAMETERS)
assert not algebra_workspace.state.errors
a = algebra_workspace.state.results
assert not any(a["cross"].values)
assert set(a["kernel_H"].values) == set(a["code"].values)
assert set(a["kernel_G"].values) == set(a["dual"].values)
print("h, packed:", a["quotient"].values.tolist())
print("g_perp coefficients:", a["g_perp"].values.tolist())
print("Integer G/H overlap counts:\n", a["overlaps"].values.reshape(4, 3))
print("All 128 masks checked: ker(H) = C; ker(G) = C_perp.")

matrices_plot = binary_panels([a["G"], a["H"]], ["G · generate C, check its dual",
                              "H · generate the dual, check C"],
                              title="Two matrices exchange roles")
FIGURES["generator-and-check-matrices"] = matrices_plot
matrices_plot.show()

tables = Workspace({"C": code["bits"].arrange(x=F.degree, y=-F.message),
                    "dual": dual["bits"].arrange(x=F.degree, y=-F.message)}, PARAMETERS)
words_plot = binary_panels([tables.state.results["C"], tables.state.results["dual"]],
                           ["16 words of C", "8 words of the dual"],
                           title="Where are the triples and their complements?", height=600)
FIGURES["code-and-dual"] = words_plot
words_plot.show()

Why does checking just the generator rows suffice? A mask orthogonal to each row
is orthogonal to every binary linear combination of those rows. The reciprocal
construction supplies three independent rows orthogonal to $G$; the finite checks
above also establish both span/kernel equalities on the entire 128-word domain.
For background, see [MIT's polynomial Hamming-code notes](https://math.mit.edu/~djk/18.310/Lecture-Notes/polynomial_hamming_codes_2007.html)
and [Sage's cyclic-code construction](https://doc.sagemath.org/html/en/reference/coding/sage/coding/cyclic_code.html).

## Seven triples reveal a projective plane

Read the columns of $H$ as vectors in $\mathbf F_2^3$. Each is nonzero, and each
occurs once. A weight-three codeword selects columns $u,v,w$ with $u+v+w=0$.
Thus its support is $\{u,v,u+v\}$: precisely the nonzero part of a two-dimensional
subspace. These seven triples are the lines of **$\mathrm{PG}(2,2)$**, the Fano plane.

In projective terminology, points are one-dimensional subspaces. Over $\mathbf F_2$
there is only one nonzero scalar, so each point has a single nonzero representative.
Zero is not a projective point. Any two distinct points determine their third
point on a line by binary vector addition.

Count the 1s in each word to select the seven triples. Order the selected words
explicitly; the measured ranks place their seven copies of the plane. The same
coefficient occurrences can move from concentric rings to those charts.

In [ ]:
def fano_supports(code, check_matrix):
    columns = check_matrix.group_by(F.degree).sum(value=F.value * 2 ** F.row)
    lines = code["words"].annotate(word=F.value).where(
        code["weights"].bind(on=F.message) == 3).select()
    ranks = lines.group_by().order_by(F.word).ranks(key=F.word)
    lines = lines.annotate(line_rank=ranks.bind(on=F.word))
    supports = Collection.grid(lines.count().scalar(), 7, axes=("line_rank", "degree"))
    supports = supports.annotate(word=lines.bind(on=F.line_rank, key=F.line_rank),
                                 vector=columns.bind(on=F.degree))
    supports = supports.with_values(bit(F.word, F.degree))
    return dict(columns=columns, lines=lines, ranks=ranks, supports=supports)


def fano_chart():
    # Keys are the nonzero vectors of F_2^3, packed with the first component low.
    # A chart changes only placement; collinearity is still a binary relation.
    return Arrangement.points(
        range(1, 8),
        [(-sqrt(3), -1), (sqrt(3), -1), (0, -1), (0, 2),
         (-sqrt(3)/2, .5), (sqrt(3)/2, .5), (0, 0)],
        keys=range(1, 8), name="A chart of the Fano plane")


def support_placements(supports, chart):
    radius = 1.2 + .45 * F.line_rank
    rings = supports.arrange(x=radius*cos(2*pi*F.degree/7),
                             y=radius*sin(2*pi*F.degree/7))
    panels = supports.arrange(
        x=6*(F.line_rank % 4) + chart.bind(on=F.vector, read=F.x),
        y=-5*(F.line_rank // 4) + chart.bind(on=F.vector, read=F.y))
    shifted = supports.annotate(degree=(F.degree + 1) % 7)
    shifted_rings = shifted.arrange(x=radius*cos(2*pi*F.degree/7),
                                    y=radius*sin(2*pi*F.degree/7))
    return rings, shifted_rings, panels


def support_turn():
    angle = 2*pi*param("time")/7
    return Motion.custom(F.sx*cos(angle)-F.sy*sin(angle),
                         F.sx*sin(angle)+F.sy*cos(angle))

In [ ]:
geometry = fano_supports(code, H)
chart = fano_chart()
rings, shifted_supports, panels = support_placements(geometry["supports"], chart)
geometry_workspace = Workspace({**geometry, "panels": panels, "chart": chart}, PARAMETERS)
assert not geometry_workspace.state.errors
f = geometry_workspace.state.results
assert set(f["columns"].values) == set(range(1, 8))
line_words = set(map(int, f["lines"].values))
dual_words = set(map(int, a["dual"].values))
assert {127-word for word in line_words} == dual_words - {0}
pair_counts = Counter(pair for word in line_words
                      for pair in combinations([j for j in range(7) if bit(word, j)], 2))
assert set(pair_counts) == set(combinations(range(7), 2))
assert set(pair_counts.values()) == {1}
print("H column labels:", f["columns"].values.tolist())
print("All 21 point pairs have exactly one line; seven complements match the nonzero dual.")

support_workspace = Workspace({"supports": rings.where(F.value == 1)}, PARAMETERS)
pack = support_workspace.set("supports", panels.where(F.value == 1), motion=Motion.arc(height=1.4))
support_workspace.capture("Seven weight-three supports, arranged by measured word ranks.")
unpack = support_workspace.undo()
support_frames, support_labels = [], []
for transition, label in ((pack, "Arrange the same coefficient occurrences"),
                           (unpack, "Undo the arrangement")):
    frames, labels = sampled(transition, "supports", label, steps=41)
    support_frames.extend(frames)
    support_labels.extend(labels)
palette = {oid: COLORS[1] if value else COLORS[2]
           for oid, value in zip(f["panels"].ids, f["panels"].values)}
support_plot = replay(support_frames, support_labels, colors_by_id=palette,
                      title="Seven support masks change their arrangement")
FIGURES["support-motion"] = support_plot
support_plot.show()

fano_plot = fano_gallery(f["panels"], f["chart"])
FIGURES["fano-code-and-dual"] = fano_plot
fano_plot.show()

The connecting strokes are a drawing convention. In this chart one projective
line is a circle; Euclidean straightness does not define the incidence. Amber
selects one codeword, and violet its dual complement. `c` and `d` above each panel
are packed seven-bit labels; the point labels pack the three coordinates of an
$H$ column. Keep those two domains distinct.

## The same cubic constructs a field

Now use $g(t)$ as the modulus for **$K=\mathbf F_2[t]/(g(t))$**, and write
$\alpha=t\bmod g$. Both offered cubics are irreducible: a cubic over a field is
reducible exactly when it has a root, and neither has a root at 0 or 1.
So $K$ has eight elements, represented by $a_0+a_1\alpha+a_2\alpha^2$.
In the default model, $\alpha^3=1+\alpha$.

This is a different quotient from the seven-slot code ring. The polynomial is
shared, but the objects have different roles. The projective plane is built by
viewing $K$ as a **three-dimensional vector space over $\mathbf F_2$**. Its seven
nonzero elements represent the projective points. This gives an incidence
isomorphism between two constructions of the plane, not an isomorphism between
the plane and a field as mathematical structures.

The next recipe performs bounded binary long division. It also checks that each
nonzero element has one multiplicative inverse before reusing the table as a field.

In [ ]:
def binary_sum(left, right, width):
    return sum(((bit(left, j) + bit(right, j)) % 2) * 2 ** j for j in range(width))


def cubic_remainder(items, modulus, highest_degree):
    """A visible, bounded long-division recipe; each subtraction is a graph node."""
    for degree in range(highest_degree, 2, -1):
        multiple = bit(F.value, degree) * modulus * 2 ** (degree - 3)
        items = items.with_values(binary_sum(F.value, multiple, highest_degree + 1))
    return items


def field_model(polynomial, columns):
    modulus = polynomial.sum(value=F.value * 2 ** F.key).scalar()
    products = Collection.grid(8, 8, axes=("left", "right"),
                               values=polynomial_product(F.left, F.right, 3, 3))
    multiply = cubic_remainder(products, modulus, 4)
    inverse_coverage = multiply.where(F.value == 1).group_by(F.left).coverage().on_keys(F.key != 0)
    field_check = inverse_coverage.exactly(1)
    multiply = multiply.require(field_check, message="A nonzero element lacks a unique inverse")
    powers = cubic_remainder(Collection.sequence(8, start=0).with_values(2 ** F.key),
                             modulus, 7).require(field_check)
    # T sends the polynomial basis (1, alpha, alpha^2) to H's first three columns.
    atlas = powers.where(F.key < 7).select().annotate(exponent=F.key)
    atlas = atlas.annotate(vector=sum(
        (sum(bit(F.value, i) * bit(columns.bind(on=i), j) for i in range(3)) % 2)
        * 2 ** j for j in range(3)))
    elements = Collection.sequence(8, start=0)
    elements = elements.annotate(square=multiply.bind(on=(F.value, F.value),
                                                     key=(F.left, F.right)))
    elements = elements.annotate(fourth=multiply.bind(on=(F.square, F.square),
                                                     key=(F.left, F.right)))
    trace = elements.with_values(binary_sum(binary_sum(F.value, F.square, 3), F.fourth, 3))
    pairs = Collection.grid(7, 7, axes=("line", "degree")).annotate(
        normal=atlas.bind(on=F.line), element=atlas.bind(on=F.degree))
    pairs = pairs.with_values(trace.bind(on=multiply.bind(
        on=(F.normal, F.element), key=(F.left, F.right))))
    incidence = pairs.where(F.value == 0)
    lines = incidence.group_by(F.line).sum(value=2 ** F.degree)
    dual_words = pairs.group_by(F.line).sum(value=F.value * 2 ** F.degree)
    return dict(multiply=multiply, inverses=inverse_coverage.counts, powers=powers, atlas=atlas, trace=trace,
                incidence=incidence, lines=lines, dual_words=dual_words)

In [ ]:
field = field_model(g, geometry["columns"])
field_workspace = Workspace(field, PARAMETERS)
assert not field_workspace.state.errors
k = field_workspace.state.results
assert k["powers"].values[-1] == 1
assert set(k["atlas"].values) == set(range(1, 8))
np.testing.assert_array_equal(k["atlas"].fields["vector"], f["columns"].values)
assert set(k["lines"].values) == line_words
assert set(k["dual_words"].values) == dual_words - {0}

rows = ["| Degree j | Field element αʲ (packed) | Polynomial coordinates | H-column coordinates |",
        "| --- | --- | --- | --- |"]
for j, value, vector in zip(k["atlas"].fields["exponent"], k["atlas"].values,
                             k["atlas"].fields["vector"]):
    coordinates = tuple(bit(int(value), i) for i in range(3))
    h_coordinates = tuple(bit(int(vector), i) for i in range(3))
    rows.append(f"| {j} | {value} | {coordinates} | {h_coordinates} |")
display(Markdown("\n".join(rows)))
print("Field trace at packed elements 0,...,7:", k["trace"].values.tolist())
print("Trace-zero line masks:", k["lines"].values.tolist())
print("Trace-one dual masks:", k["dual_words"].values.tolist())

The coordinate map is explicit:

$$T(a_0+a_1\alpha+a_2\alpha^2)=a_0H_0+a_1H_1+a_2H_2,$$

where $H_j$ is column $j$. The table checks $T(\alpha^j)=H_j$ for all seven
degrees. Consequently $Hc^T=T(\sum_j c_j\alpha^j)$: passing the matrix checks
means $c(\alpha)=0$, equivalently divisibility by $g$. A code relation and a field
relation now describe the same incidence.

For another construction of the lines, use the field trace
$\operatorname{Tr}(z)=z+z^2+z^4$. For each nonzero $a\in K$ form

$$L_a=\{z\in K^*: \operatorname{Tr}(az)=0\}.$$

The seven packed masks of these sets match the code's seven triples; the trace-one
masks match its seven nonzero dual words. This comparison checks every one of the
49 line/point pairs after identifying each line by its support, rather than matching
two pictures by eye. In the default model, the initial trace-zero line is
$\{\alpha,\alpha^2,\alpha^4\}$.

Multiplication by $\alpha$ is an invertible $\mathbf F_2$-linear map, so it sends
lines to lines. On exponents it is just $j\mapsto j+1\pmod7$. Its seven-step action
is a **Singer cycle**. The extension-field construction also works for larger
projective spaces; [Sage's Singer construction](https://doc.sagemath.org/html/en/reference/combinat/sage/combinat/designs/difference_family.html#sage.combinat.designs.difference_family.singer_difference_set)
describes that broader setting.

**Predict:** where will the amber triple be after one turn? After seven? Scrub
to an exact step boundary to check collinearity. The labels are packed **field**
coordinates, not seven-bit codeword labels. A changing label records the two
discrete endpoint values; it is never a fractional field element.

In [ ]:
def multiply_by_alpha(points, field):
    moved = points.with_values(field["multiply"].bind(
        on=(2, F.value), key=(F.left, F.right)))
    return moved.annotate(
        exponent=field["atlas"].bind(on=F.value, key=F.value, read=F.exponent),
        vector=field["atlas"].bind(on=F.value, key=F.value, read=F.vector))


def field_placements(points, chart):
    ring = points.arrange(x=2*cos(2*pi*F.exponent/7), y=2*sin(2*pi*F.exponent/7))
    plane = points.arrange(x=chart.bind(on=F.vector, read=F.x),
                           y=chart.bind(on=F.vector, read=F.y))
    return ring, plane

In [ ]:
points = field["atlas"].annotate(seed_trace=field["trace"].bind(on=F.value))
ring, plane = field_placements(points, chart)
cycle_workspace = Workspace({"ring": ring.where(F.seed_trace == 0),
                             "plane": plane.where(F.seed_trace == 0)}, PARAMETERS)
seed = cycle_workspace.state.results["ring"].source
ring_frames, plane_frames, cycle_labels = [], [], []
ring_transitions, plane_transitions = [], []
for step in range(1, 8):
    points = multiply_by_alpha(points, field)
    ring, plane = field_placements(points, chart)
    turn = cycle_workspace.set("ring", ring.where(F.seed_trace == 0), motion=support_turn())
    crossing = cycle_workspace.set("plane", plane.where(F.seed_trace == 0), motion=Motion.arc(height=.4))
    ring_transitions.append(turn)
    plane_transitions.append(crossing)
    rf, labels = sampled(turn, "ring", f"Multiply by α · step {step}/7", steps=17)
    pf, _ = sampled(crossing, "plane", "", steps=17)
    ring_frames.extend(rf)
    plane_frames.extend(pf)
    cycle_labels.extend(labels)
np.testing.assert_array_equal(cycle_workspace.state.results["ring"].source.values, seed.values)
np.testing.assert_array_equal(cycle_workspace.state.results["ring"].source.positions, seed.positions)
assert cycle_workspace.state.results["ring"].source.ids == seed.ids
cycle_workspace.capture("A complete Singer cycle returns all seven occurrences to their starting values and positions.")

# Undo the two recorded chart updates for the last field action. They are two
# workspace edits displayed at the same progress; no interpolated frame is an input.
undo_plane = cycle_workspace.undo()
undo_ring = cycle_workspace.undo()
rf, labels = sampled(undo_ring, "ring", "Undo the seventh turn", steps=17)
pf, _ = sampled(undo_plane, "plane", "", steps=17)
ring_frames.extend(rf)
plane_frames.extend(pf)
cycle_labels.extend(labels)
np.testing.assert_array_equal(ring_transitions[-1].frame("ring", .25).positions,
                               undo_ring.frame("ring", .75).positions)
cycle_plot = linked_field_motion(ring_frames, plane_frames, cycle_labels,
                                 supports=f["supports"], chart=f["chart"], seed=seed)
FIGURES["field-and-projective-motion"] = cycle_plot
cycle_plot.show()

## What makes this error-correcting?

A corrupted word $r=c+e$ has syndrome $Hr^T=He^T$, because $Hc^T=0$.
For a single error in slot $j$, this is column $H_j$. The seven columns are
distinct and nonzero, so they identify the erroneous slot. Zero syndrome gets an
explicit no-error candidate. We use a coverage measurement to require exactly one
candidate before letting its value drive the correction.

This finite rule corrects every codeword with at most one changed bit. The full
covering and coset-motion investigation is a [planned follow-up](../docs/lessons/CODES_AND_DISCOVERY_PATHS.md#c--error-correction-becomes-a-covering-problem).
[MIT's matrix-code notes](https://math.mit.edu/~djk/18.310/Lecture-Notes/matrix_hamming_codes_2007.html)
explain the syndrome argument. Inferring a correction is different from undo:
undo knows the captured earlier state; a decoder sees only the received word.

In [ ]:
def single_error_rule(syndromes):
    # Zero syndrome has its own explicit no-error candidate.
    errors = Collection.sequence(8, start=0, name="No error or one changed bit")
    errors = errors.annotate(error=F.value).with_values((2 ** F.value) // 2)
    labels = errors.annotate(syndrome=syndromes.bind(on=F.value))
    matches = Collection.grid(8, 8, axes=("syndrome", "error"),
                              values=labels.bind(on=F.error, key=F.error))
    matches = matches.where(labels.bind(on=F.error, key=F.error, read=F.syndrome)
                            == F.syndrome)
    coverage = matches.group_by(F.syndrome).coverage()
    correction = coverage.unique()
    return dict(errors=labels, matches=matches, coverage=coverage.counts,
                correction=correction)

In [ ]:
decoder = single_error_rule(checks_H["syndromes"])
message = code["words"].bind(on=param("message"))
received = binary_sum(message, param("error_mask"), 7)
syndrome = checks_H["syndromes"].bind(on=received)
correction = decoder["correction"].bind(on=syndrome)
corrected = binary_sum(received, correction, 7)
slots = Collection.sequence(7, start=0)
transmission = {
    "sent": slots.with_values(bit(message, F.key)).arrange(x=F.key, y=0),
    "received": slots.with_values(bit(received, F.key)).arrange(x=F.key, y=0),
    "decoded": slots.with_values(bit(corrected, F.key)).arrange(x=F.key, y=0),
}
decoding_workspace = Workspace({**decoder, **transmission}, PARAMETERS)
assert not decoding_workspace.state.errors
e = decoding_workspace.state.results
assert e["coverage"].values.tolist() == [1]*8
print("Syndrome → measured correction mask:", e["correction"].values.tolist())
returned = np.array_equal(e["sent"].values, e["decoded"].values)
print("Decoded to the sent word:", returned)
if PARAMETERS["error_mask"].bit_count() <= 1:
    assert returned
decode_plot = binary_panels([e["sent"], e["received"], e["decoded"]],
                            ["Sent", "Received", "Decoded"],
                            title="A measured syndrome selects the correction", height=330)
FIGURES["single-error-correction"] = decode_plot
decode_plot.show()

## Break an assumption; retain a witness

1. **Omit the reciprocal.** The quotient $h$ itself need not generate the dual.
2. **Use $1+x+x^2+x^3$.** Its four shifted rows still span a linear code, but
   cyclic closure fails. Its factor search must report failure.
3. **Change two bits.** A one-error decoder can return a different codeword.
   It does not recover the actual corruption history.
4. **Use $t^3+1$ as a field modulus.** It has a root at 1. The inverse-coverage
   check exposes nonzero elements that lack inverses.

In [ ]:
_, wrong_H = generator_rows(coefficients(quotient.scalar()), 3)
_, wrong_parities = cross_parities(G, wrong_H, 4, 3)
wrong = wrong_parities.evaluate(**PARAMETERS)
bad_pairs = [(int(i), int(j)) for i, j, value in zip(wrong.fields["left"],
                                                    wrong.fields["right"], wrong.values) if value]
assert bad_pairs

bad_g = coefficients(15)
_, bad_G = generator_rows(bad_g, 4)
bad_code = span(bad_G, 4)
_, bad_quotient, _ = dual_polynomial(bad_g)
broken = Workspace({"words": bad_code["words"], "quotient": bad_quotient})
assert "quotient" in broken.state.errors and "words" not in broken.state.errors
bad_words = set(map(int, broken.state.results["words"].values))
rolled_68 = ((68 << 1) & 127) | (68 >> 6)
assert 68 in bad_words and rolled_68 == 9 and 9 not in bad_words

syndromes = checks_H["syndromes"].evaluate(**PARAMETERS)
syndrome_by_word = dict(zip(map(int, syndromes.fields["word"]), map(int, syndromes.values)))
error_by_syndrome = dict(zip(map(int, e["correction"].fields["syndrome"]), map(int, e["correction"].values)))
two_error_word = 3  # Transmit zero, then change degrees 0 and 1.
wrong_decoded = two_error_word ^ error_by_syndrome[syndrome_by_word[two_error_word]]
assert wrong_decoded != 0 and wrong_decoded in set(a["code"].values)

bad_field = field_model(coefficients(9), geometry["columns"])
field_failure = Workspace({"inverses": bad_field["inverses"], "multiply": bad_field["multiply"]}, PARAMETERS)
assert "multiply" in field_failure.state.errors
print("Without reversal, odd G/H row overlaps:", bad_pairs)
print("Noncyclic stencil: word 68 shifts to missing word", rolled_68)
print("Two errors on the zero word: received 3 decodes to", wrong_decoded)
print("Reducible cubic inverse counts:", field_failure.state.results["inverses"].values.tolist())

## Continue the investigation

- Change `generator` from 11 to 13. The field-coordinate dictionary changes.
  Which objects are equal, and which are isomorphic after relabeling?
- Before replaying, choose two points in the projective chart and predict the
  third on their line. Use either binary vector addition or addition in the field,
  passing through the displayed coordinate dictionary.
- Change the message and the error mask. Try no error and all seven single
  errors, then compare a two-error corruption with the decoder's explanation.
- Reorder the $H$ occurrences or the measured column table before binding them.
  The logical keys still determine the result; their storage order should not.
- Inspect a parity-zero output with two integer contributors. What would be lost
  if the program retained only a Boolean result?

From a blank workspace, the choices here were: coefficient arithmetic and order;
a generator stencil; which axes to retain in a measurement; a condition for unique
reuse; the coordinate dictionary; and which exact action to associate with motion.
These are candidates for future novice authoring controls. The notebook exercises
existing primitives; it adds no polynomial/field engine to the core.

In [ ]:
# A specific cancellation, including the original product occurrences.
cancelled = next((m, j) for m, j, value in zip(c["overlaps"].fields["message"],
                    c["overlaps"].fields["degree"], c["overlaps"].values) if value == 2)
contributors = c["overlaps"].contributor_ids(cancelled)
assert len(contributors) == 2
assert c["overlaps"].contributor_ids((0, 0)) == ()
evidence = {
    "parameters": PARAMETERS,
    "coefficient_order": "low degree first; packed integer labels",
    "cancelled_output": {"message": int(cancelled[0]), "degree": int(cancelled[1]),
                         "integer_count": 2, "parity": 0, "contributors": list(contributors),
                         "count_node": c["overlaps"].node},
    "factor_candidate": {"value": int(a["quotient"].values[0]),
                         "contributors": list(a["quotient"].contributor_ids(()))},
    "field_to_H_coordinates": [
        {"degree": int(j), "field": int(value), "H_column": int(vector)}
        for j, value, vector in zip(k["atlas"].fields["exponent"], k["atlas"].values,
                                    k["atlas"].fields["vector"])],
    "counterexamples": {"wrong_reciprocal_pairs": bad_pairs,
                         "noncyclic_shift": [68, 9], "two_error_decode": [3, wrong_decoded]},
    "scope": "Exact checks for the chosen seven-slot binary code; paths are presentation only.",
}
(OUTPUT / "explanations.json").write_text(json.dumps(evidence, indent=2, allow_nan=False))

for workspace, filename in ((generator_workspace, "generator-workspace.json"),
                            (algebra_workspace, "algebra-workspace.json"),
                            (geometry_workspace, "geometry-workspace.json"),
                            (field_workspace, "field-workspace.json"),
                            (support_workspace, "support-workspace.json"),
                            (cycle_workspace, "cycle-workspace.json"),
                            (decoding_workspace, "decoding-workspace.json")):
    reopened = compact_workspace(workspace, filename)
    assert set(reopened.state.results) == set(workspace.state.results)
restored = Workspace.from_json((OUTPUT / "cycle-workspace.json").read_text())
replayed = restored.redo()
np.testing.assert_array_equal(replayed.frame("ring", .25).positions,
                               ring_transitions[-1].frame("ring", .25).positions)
save_figures(OUTPUT, FIGURES)

checks = {"code_words": 16, "dual_words": 8, "point_pairs_checked": len(pair_counts),
          "field_products": 64, "field_line_incidences": 49,
          "full_cycle_steps": 7, "field_motion_frames": len(ring_frames),
          "support_motion_frames": len(support_frames), "generator_motion_frames": len(generator_frames),
          "code_weights": {int(w): int(n) for w, n in Counter(c["weights"].values).items()},
          "pending_cycle_redo": cycle_workspace.can_redo}
(OUTPUT / "checks.json").write_text(json.dumps(checks, indent=2))
print("Saved definitions, contributor explanations, captured workspaces, and offline Plotly HTML to", OUTPUT)

## Recorded videos

The first video records the support rearrangement and its undo. The second is the
projective chart of the field action. The MP4 renderer displays the captured points
and their discrete endpoint labels; the interactive chart above adds the incidence
strokes and the synchronized cyclic view. Both videos use the same sampled frames
as the interactive figures. Reopening the saved workspace replays captured data.

In [ ]:
support_video = write_mp4(support_frames, OUTPUT / "supports-and-undo.mp4",
                          labels=support_labels, title="Seven code supports · rearrange and undo",
                          fps=20, size=(1000, 620))
field_video = write_mp4(plane_frames, OUTPUT / "projective-field-action.mp4",
                        labels=[label.replace("α", "alpha") for label in cycle_labels],
                        title="Multiplication by alpha · projective chart",
                        fps=20, size=(1000, 620))
display(Video(str(support_video), embed=True, width=900))
display(Video(str(field_video), embed=True, width=900))